# 🔥 Experiment 007b: Gemma-2B-it LoRA SFT (Chat-Template Fixed)

**Goal:** Fine-tune Gemma-2B-it to emit `[ROUTE_*]` tokens via LoRA.

**Fix from 007:** Training data now wrapped in Gemma's `<start_of_turn>` chat template
so the model learns to route from the same prompt format the harness uses.

**Baseline (Exp 006):** Vanilla Gemma-2B-it scored **52.1% Direct** (0% Route).

## Setup
1. **Runtime → Change runtime type → T4 GPU**
2. Add `HF_TOKEN` to Colab Secrets (🔑 icon in sidebar)
3. Run all cells in order
4. Results save to Google Drive

## 1. Install Dependencies

In [1]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes huggingface_hub

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.8 MB/s eta 0:00:00
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## 2. Authenticate with HuggingFace

In [2]:
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print('✅ Logged in via Colab Secrets')
except Exception:
    login()
    print('✅ Logged in manually')

✅ Logged in via Colab Secrets


## 3. Clone Repo & Load Chat-Template-Wrapped Training Data

In [3]:
import os

REPO_DIR = '/content/tagzeit-gemma-2b'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/mgosal/tagzeit-gemma-2b.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

# ⚠️ KEY FIX: Use Gemma-formatted data (chat template wrapped)
# NOT the raw format that caused 007's 0% E2E result
train_file = f'{REPO_DIR}/data/train/train_routed_gemma.jsonl'
eval_file = f'{REPO_DIR}/data/eval/eval_routed_gemma.jsonl'

assert os.path.exists(train_file), f'Training data not found at {train_file}'
assert os.path.exists(eval_file), f'Eval data not found at {eval_file}'

!wc -l {train_file} {eval_file}
print()
print('Sample (first record):')
!head -1 {train_file} | python3 -c "import json,sys; print(json.loads(sys.stdin.read())['text'][:300])"

Cloning into '/content/tagzeit-gemma-2b'...
remote: Enumerating objects: 269, done.
remote: Counting objects: 100% (269/269), done.
remote: Compressing objects: 100% (198/198), done.
remote: Total 269 (delta 106), reused 213 (delta 54), pack-reused 0 (from 0)
Receiving objects: 100% (269/269), 400.37 KiB | 2.13 MiB/s, done.
Resolving deltas: 100% (106/106), done.
   5640 /content/tagzeit-gemma-2b/data/train/train_routed_gemma.jsonl
    303 /content/tagzeit-gemma-2b/data/eval/eval_routed_gemma.jsonl
   5943 total

Sample (first record):
<bos><start_of_turn>user
What is 58 + 17?<end_of_turn>
<start_of_turn>model
[NO_ROUTE] This is base-10 arithmetic, not temporal.<end_of_turn>



## 4. Domain Token Registry

104 typed tokens for the Route-to-Luxon grammar.

In [4]:
import math
import hashlib
import numpy as np

ROUTING_TOKENS = [
    "[ROUTE_TIME_ADD]", "[ROUTE_TIME_SUB]",
    "[ROUTE_DURATION_BETWEEN]", "[ROUTE_CALENDAR_SHIFT]",
    "[ROUTE_TIMEZONE_CONVERT]",
]
HEAD_TOKENS = [
    "[HEAD_TIME]", "[HEAD_DURATION]", "[HEAD_PLUS]",
    "[HEAD_MINUS]", "[HEAD_DATE]", "[HEAD_DATETIME]",
]
ARG_HOUR_TOKENS = [f"[ARG_HOUR_{h:02d}]" for h in range(24)]
ARG_MIN_TOKENS = [f"[ARG_MIN_{m:02d}]" for m in range(60)]
REF_TOKENS = [f"[REF_{i}]" for i in range(1, 10)]

ALL_SPECIAL_TOKENS = (
    ROUTING_TOKENS + HEAD_TOKENS +
    ARG_HOUR_TOKENS + ARG_MIN_TOKENS +
    REF_TOKENS
)
print(f"Total domain tokens: {len(ALL_SPECIAL_TOKENS)}")

def sinusoidal_encoding(position, d_model, max_period=10000):
    encoding = np.zeros(d_model)
    for i in range(0, d_model, 2):
        div_term = max_period ** (i / d_model)
        encoding[i] = math.sin(position / div_term)
        if i + 1 < d_model:
            encoding[i + 1] = math.cos(position / div_term)
    return encoding

def compute_geometric_init(token, d_model, existing_mean, existing_std):
    def _seed(s): return int(hashlib.md5(s.encode()).hexdigest()[:8], 16)

    if token.startswith("[ARG_HOUR_"):
        hour = int(token.split("_")[-1].rstrip("]"))
        base = np.zeros(d_model)
        for i in range(0, d_model, 2):
            base[i] = math.sin(hour / (10000 ** ((i//2) / (d_model//2))))
        return existing_mean + base * existing_std * 0.5
    elif token.startswith("[ARG_MIN_"):
        minute = int(token.split("_")[-1].rstrip("]"))
        base = np.zeros(d_model)
        for i in range(1, d_model, 2):
            base[i] = math.sin(minute / (10000 ** ((i//2) / (d_model//2))))
        return existing_mean + base * existing_std * 0.5
    elif token.startswith("[ROUTE_"):
        rng = np.random.default_rng(_seed(token))
        perturbation = rng.normal(0, existing_std * 0.1, d_model)
        offset = sinusoidal_encoding(50, d_model) * existing_std * 0.2
        return existing_mean + perturbation + offset
    else:
        rng = np.random.default_rng(_seed(token))
        return existing_mean + rng.normal(0, existing_std * 0.1, d_model)

print("✅ Token registry and geometric init loaded")

Total domain tokens: 104
✅ Token registry and geometric init loaded


## 5. Load Model + Apply LoRA

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

MODEL_ID = "google/gemma-2-2b-it"

print(f"Loading {MODEL_ID}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Register domain tokens
original_vocab_size = len(tokenizer)
num_added = tokenizer.add_special_tokens(
    {"additional_special_tokens": list(ALL_SPECIAL_TOKENS)}
)
model.resize_token_embeddings(len(tokenizer))
print(f"Registered {num_added} domain tokens (vocab: {original_vocab_size} → {len(tokenizer)})")

# Geometric embedding init
if num_added > 0:
    with torch.no_grad():
        embed_layer = model.get_input_embeddings()
        lm_head = model.get_output_embeddings()
        existing_embeds = embed_layer.weight[:original_vocab_size].float()
        existing_mean = existing_embeds.mean(dim=0).cpu().numpy()
        existing_std = existing_embeds.std().item()
        d_model = embed_layer.weight.shape[1]

        for token in ALL_SPECIAL_TOKENS:
            token_id = tokenizer.convert_tokens_to_ids(token)
            if token_id is None or token_id < original_vocab_size:
                continue
            init_vec = compute_geometric_init(token, d_model, existing_mean, existing_std)
            init_tensor = torch.tensor(init_vec, dtype=torch.float16)
            embed_layer.weight[token_id] = init_tensor
            if lm_head is not None and lm_head.weight.shape[0] > token_id:
                lm_head.weight[token_id] = init_tensor

    print(f"Applied geometric init for {num_added} tokens (d={d_model})")

# LoRA
peft_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
print("✅ Model loaded with LoRA")

Loading google/gemma-2-2b-it...


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Registered 104 domain tokens (vocab: 256000 → 256104)
Applied geometric init for 104 tokens (d=2304)
trainable params: 83,066,880 || all params: 2,697,648,384 || trainable%: 3.0792
✅ Model loaded with LoRA


## 6. Load Dataset

In [6]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={"train": train_file, "eval": eval_file},
)
train_dataset = dataset["train"]
eval_dataset = dataset["eval"]

print(f"Train: {len(train_dataset)} samples")
print(f"Eval:  {len(eval_dataset)} samples")
print(f"\nSample (truncated):\n{train_dataset[1]['text'][:300]}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating eval split: 0 examples [00:00, ? examples/s]

Train: 5640 samples
Eval:  303 samples

Sample (truncated):
<bos><start_of_turn>user
I put the roast in at 09:58. It needs 17 minutes. When is it done?<end_of_turn>
<start_of_turn>model
[ROUTE] [ROUTE_TIME_ADD] [HEAD_TIME] [ARG_HOUR_09] [ARG_MIN_58] [HEAD_DURATION] [ARG_MIN_17] [/ROUTE]<end_of_turn>



## 7. Mount Google Drive & Configure Training

In [7]:
from trl import SFTTrainer, SFTConfig
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/tagzeit/007b-gemma2-2b-lora-chatfmt'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output: {OUTPUT_DIR}")

Mounted at /content/drive
Output: /content/drive/MyDrive/tagzeit/007b-gemma2-2b-lora-chatfmt


## 8. Train! 🚀

**Config notes (fixed from 007):**
- Batch=2, grad_accum=4 (effective=8) — avoids T4 OOM
- Eval batch=1 — eval was the OOM trigger in 007
- Checkpoints every 250 steps — never lose >15min of work
- 2,500 max steps — review results before going further

In [8]:
import glob

MAX_STEPS = 2500

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    save_strategy="steps",
    save_steps=250,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    max_steps=MAX_STEPS,
    warmup_steps=100,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    optim="adamw_8bit",

    fp16=True,
    bf16=False,

    logging_steps=10,
    eval_strategy="steps",
    eval_steps=500,
    per_device_eval_batch_size=1,
    report_to="none",

    max_length=256,
    packing=True,
    dataset_text_field="text",

    seed=3407,
)

def formatting_func(example):
    return example["text"]

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    formatting_func=formatting_func,
)

# Resume from checkpoint if available
checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"))
if checkpoints:
    latest = checkpoints[-1]
    print(f"🔄 Resuming from {latest}")
    trainer.train(resume_from_checkpoint=latest)
else:
    print("🚀 Starting fresh training run")
    trainer.train()

# Save final adapter
print(f"\n✅ Training complete! Saving to {OUTPUT_DIR}")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

Applying formatting function to train dataset:   0%|          | 0/5640 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/5640 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5640 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/5640 [00:00<?, ? examples/s]

Applying formatting function to eval dataset:   0%|          | 0/303 [00:00<?, ? examples/s]

...

Step,Training Loss,Validation Loss
500,0.396398,0.391475
1000,0.360115,0.387376
1500,0.275015,0.506589
2000,0.252565,0.573665
2500,0.237464,0.608134


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(



✅ Training complete! Saving to /content/drive/MyDrive/tagzeit/007b-gemma2-2b-lora-chatfmt


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


('/content/drive/MyDrive/tagzeit/007b-gemma2-2b-lora-chatfmt/tokenizer_config.json',
 '/content/drive/MyDrive/tagzeit/007b-gemma2-2b-lora-chatfmt/chat_template.jinja',
 '/content/drive/MyDrive/tagzeit/007b-gemma2-2b-lora-chatfmt/tokenizer.json')

## 9. Sanity Check — Does it route?

In [9]:
model.eval()
test_prompts = [
    "Meeting starts at 14:30, lasts 45 minutes. When does it end?",
    "The train arrived at 09:15. Journey was 25 minutes. Departure time?",
    "What is 42 + 18?",
]

for prompt in test_prompts:
    # Use chat template — same format as training data
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=64,
            do_sample=False, temperature=1.0
        )
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=False
    )
    print(f"Q: {prompt}")
    print(f"A: {response.strip()}")
    print()

Q: Meeting starts at 14:30, lasts 45 minutes. When does it end?
A: [ROUTE] [ROUTE_TIME_ADD] [HEAD_TIME] [ARG_HOUR_14] [ARG_MIN_30] [HEAD_DURATION] [ARG_MIN_45] [/ROUTE]<end_of_turn>

Q: The train arrived at 09:15. Journey was 25 minutes. Departure time?
A: [ROUTE] [ROUTE_TIME_SUB] [HEAD_TIME] [ARG_HOUR_09] [ARG_MIN_15] [HEAD_DURATION] [ARG_MIN_25] [/ROUTE]<end_of_turn>

Q: What is 42 + 18?
A: [NO_ROUTE] This is base-10 arithmetic, not temporal.<end_of_turn>



## 10. Download Adapter Weights

Weights are saved to Google Drive at:
`My Drive/tagzeit/007b-gemma2-2b-lora-chatfmt/`

To evaluate locally:
```bash
# Copy weights from Drive to your project
cp -r ~/path/to/007b-gemma2-2b-lora-chatfmt \
   experiments/route-to-luxon/weights/007b-gemma2-2b-lora-chatfmt

# Run evaluation
HF_HOME="$(pwd)/.hf_cache" .venv/bin/python tools/validate_route.py \
  --model_id experiments/route-to-luxon/weights/007b-gemma2-2b-lora-chatfmt \
  --backend torch
```

In [10]:
print("Saved adapter files:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(path):
        size = os.path.getsize(path)
        print(f"  {f:40s} ({size/1e6:.1f} MB)" if size > 1e6 else f"  {f:40s} ({size} B)")
    else:
        print(f"  {f:40s} (dir)")

Saved adapter files:
  README.md                                (1491 B)
  adapter_config.json                      (1052 B)
  adapter_model.safetensors                (2692.6 MB)
  chat_template.jinja                      (591 B)
  checkpoint-1000                          (dir)
  checkpoint-1250                          (dir)
  checkpoint-1500                          (dir)
  checkpoint-1750                          (dir)
  checkpoint-2000                          (dir)
  checkpoint-2250                          (dir)
  checkpoint-250                           (dir)
  checkpoint-2500                          (dir)
  checkpoint-500                           (dir)
  checkpoint-750                           (dir)
  tokenizer.json                           (34.4 MB)
  tokenizer_config.json                    (2549 B)
  training_args.bin                        (5649 B)
